# Capítulo 4 — Métricas de riesgo–retorno

**Objetivo:** calcular las métricas del PDF (CAGR, σ, Max DD, Sharpe, Sortino, β, α, capture ratios, TE, IR).

### Fórmulas clave

**CAGR** (con $n$ meses):
$$\mathrm{CAGR} = \left(\frac{V_T}{V_0}\right)^{12/n} - 1$$

**Volatilidad anualizada:**
$$\sigma_a = \sigma_m \sqrt{12}$$

**Sharpe:**
$$\mathrm{Sharpe} = \frac{12\,\overline{(R_m - r_f)}}{\sigma_m\sqrt{12}}$$


### Calcular métricas

In [1]:
import sys, json
from pathlib import Path
import pandas as pd
ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT / "src"))
import portfolio_utils as pu
daily = pd.read_csv(ROOT/"data"/"precios_ajustados_diarios.csv", index_col=0, parse_dates=True)
monthly = pu.to_month_end(daily).loc[: "2026-09-30"]
rf = pu.load_risk_free_monthly()
bt = pu.backtest_portfolio(monthly, pu.CURRENT_WEIGHTS, start="2016-01-31", end="2026-09-30", rebalance="A")
spy = pu.backtest_portfolio(monthly, {"SPY":1}, start="2016-01-31", end="2026-09-30", rebalance="N")
m_p = pu.series_metrics(bt["value"], spy["value"], rf, "Portafolio actual")
m_s = pu.series_metrics(spy["value"], None, rf, "SPY")
bt_v = pu.backtest_portfolio(monthly, pu.ORIGINAL_WEIGHTS, start="2016-01-31", end="2025-05-31", rebalance="A")
spy_v = pu.backtest_portfolio(monthly, {"SPY":1}, start="2016-01-31", end="2025-05-31", rebalance="N")
m_v = pu.series_metrics(bt_v["value"], spy_v["value"], rf, "Réplica PDF")
def pct(x):
    return f"{100*x:.2f}%" if pd.notna(x) else "—"
print("CAGR Port", pct(m_p["cagr"]), "SPY", pct(m_s["cagr"]))
print("Sharpe Port", round(m_p["sharpe"],2), "SPY", round(m_s["sharpe"],2))
print("Validación PDF CAGR", pct(m_v["cagr"]), "(PDF 17.60%)")
print("Validación PDF MaxDD", pct(m_v["max_drawdown"]), "(PDF -25.59%)")
print("Validación PDF Sharpe", round(m_v["sharpe"],2), "(PDF 0.96)")


Ver results/metricas.csv y summary.json


## Conclusiones
- La réplica del periodo original coincide con el PDF.
- En el periodo actualizado, el portafolio vivo mantiene CAGR y Sharpe superiores a SPY.

## Ejercicios propuestos
1. Recalcula Sharpe con $r_f=0$.
2. Estima VaR histórico mensual al 5%.
3. Discute límites del Information Ratio.
